### import 및 함수

In [1]:
import pandas as pd
import numpy as np
import re, os
from os.path import join
import warnings
warnings.simplefilter("ignore")
import functions


# 상환개시일회차와 구간회차 맞추기부터
# 상환개시일로부터 무조건 1회차 시작, 회차별 금액은
    # 유예기간 무관, 시작회차1 > 1 : 시작회차1전까지는 0원
    # 유예기간 있고, 시작회차1 상환액이 없는 경우 : 구간회차 - 유예기간(6보다 큰 경우도 있음)
    
def 신용회복스케줄(row) :
    """
    신용회복 확정건을 대상으로
    신용회복 상환일정표2와 동일한 표 만들기(납입회차, 연체회차 그대로 적용)
    단, 구간이 10을 넘어가는 경우는 스케줄 짤림
    원금균등은 짤린 회차별 평균값을 만들수는 있으나, 원리금균등은 불가능
    
    * 작성할 함수 내용
    (n은 1~10)
    작업 A : [시작회차n,종료회차n,채무조정2월상환액n]을 행변환
    [시작회차n,종료회차n,채무조정2월상환액n] 중에서 월상환액이 0인 것은 제외하고
    처음시작회차부터 마지막 종료회차까지 실상환 스케줄을 회차별로 행으로 작성
    (참고 : n=5일때 상환액이 0이라면 6부터는 모두 0임, n=1일때 상환액이 0이라면 n=2일때는 상환액이 반드시 있음, 상환액이 중간에 0이되는 경우는 없음)
    유예기간 있고, 채무조정2월상환액1 상환액과 이자율이 없는 경우 : 모든 시작회차n, 종료회차n의 값에 유예기간을 빼줘야 함.(단 다계좌는 모든 계좌가 이럴 때만)
    
    
    작업 B : 상환예정일 작성
    상환개시일부터 1회차이며, 월마다 +1 (날짜가 해당월의 마지막날짜보다 큰 경우는 마지막 날짜로 변환하여 작성)
    작업A의 결과 실변제 첫 회차가 1보다 큰 경우(예컨대 27회부터 변제시작), 1회차부터 26회차까지 납입액은 0원으로 작성
    
    작업 C : 상환액구분 작성
    회차 <= 유예기간 : 유예
    회차 > 유예기간 : 정상
    
    작업 D : 상환여부
    (회차 <= 납입회차) & 상환예정일<=cotoff : 납입
    (회차 <= 납입회차) & 상환예정일>cotoff : 선납입
    연체회차 > 0 : [납입회차 다음 ~ 상환예정일이 컷오프일이하인 회차]까지는 연체
    
    
    * 추가작업
    - 작업D에서 연체회차와 [납입회차 다음 ~ 상환예정일이 컷오프일이하인 회차]의 수가 같은지 체크
    - 작업D에서 납입한 회차의 월상환액합계가 총납입금액과 같은지
    -> 다른 경우, 해당건의 채무자키, 신용회복키, 차이값을 엑셀파일로 출력해서 확인할 수 있게 한다.)
    
    """
    당사스케줄 = [] # 신용회복키, 회차, 상환예정일, 상환액, 상환액구분, 상환여부
    상환개시일 = row["상환개시일"]
    구간회차보정 = row["구간회차보정"]
    구간별상환액 = {}

    for n in range(1, 11):
        시작회차 = row[f"시작회차{n}"]
        종료회차 = row[f"종료회차{n}"]
        월상환액 = row[f"채무조정2월상환액{n}"]

        if pd.isna(월상환액) or 월상환액 == 0:
            continue
        if pd.isna(시작회차) or pd.isna(종료회차):
            continue

        시작회차 = int(시작회차) - 구간회차보정
        종료회차 = int(종료회차) - 구간회차보정
        for 회차 in range(max(1, 시작회차), 종료회차 + 1):
            구간별상환액[회차] = 월상환액

    if not 구간별상환액:
        return 당사스케줄

    최대회차 = max(구간별상환액)
    음수회차수 = 0
    음수회차예정일 = 상환개시일 - pd.DateOffset(months=1)
    while 음수회차예정일 > 거래일:
        음수회차수 += 1
        음수회차예정일 -= pd.DateOffset(months=1)

    for 회차 in range(-음수회차수, 최대회차 + 1):
        if 회차 == 0:
            continue

        경과개월수 = 회차 - 1 if 회차 > 0 else 회차
        상환예정일 = 상환개시일 + pd.DateOffset(months=경과개월수)
        상환액 = 구간별상환액.get(회차, 0)
        당사스케줄.append({
            "신용회복키": row["신용회복키"],
            "회차": 회차,
            "상환예정일": 상환예정일,
            "상환액": 상환액,
        })

    return 당사스케줄


### 파일읽기 및 전처리

In [7]:
# 기준데이터 읽기 
# 약식 병합의 경우 정식거래일 ~ 약식거래일 사이에 확정되어 기간스케줄이 변경되었는데 업데이트 되지 않았다면 오차 있을 수 있음
##############################
company = "솔림"  # 솔림 or 대성
cutoff = "251231" # 폴더명
실제거래일 = cutoff # 컷오프일과 실제 신복 컷오프일(거래일)이 다른 경우 'yymmdd'값을 작성, 
                    # 계좌별진행상태 거래일을 사용하다가 그 값이 수기작성으로 다시 바꿈
                    # ex)1/2일다운->1/1일이 거래일 but 1/1은 휴일이므로 12/31일이 실제 거래일
path_credits_key = r"C:\Users\DATA\Desktop\p4_신복확정not매각예정_251231.xlsx" # 전체 대상일 경우 ""
##############################
path_base = join(r"D:\3.자산\전산 dataset",company,cutoff,"parquet")
path_기준데이터 = functions.키워드로파일명찾기(폴더=path_base, 포함키워드="신용회복_기준데이터", 전체경로=True, 여러파일허용=False)

credits_ori = pd.read_parquet(path_기준데이터)

# 분석 대상만 남기기
if path_credits_key != "" : 
    credits_key = pd.read_excel(path_credits_key, dtype=str)[["신용회복키"]]
    credits = credits_ori[credits_ori.신용회복키.isin(credits_key.신용회복키)].copy()
else : 
    credits = credits_ori.copy()
    
    # 확정건만 남기기
credits = credits[(credits["신청인진행상태"]=="정상이행") & (credits["계좌진행상태내용"]=="채무조정포함")].copy()

print(f'스케줄작성대상 건수 : {len(credits)}건')
신용회복키없는건 = credits[(credits.신용회복키=="") | credits.신용회복키.isna()]
if 신용회복키없는건.empty :
    print(f'○ 신용회복키 없는 건 없음')
else : 
    print(f'★ 신용회복키 없는 건 있음 : {len(신용회복키없는건)}건')
    display(신용회복키없는건)
    
# 거래일
거래일 = pd.to_datetime(실제거래일, format="%y%m%d") # 폴더명과 같겠지만 혹시몰라서

# 타입설정
credits["상환개시일"] = pd.to_datetime(credits["상환개시일"])
credits['유예기간'] = credits['유예기간'].astype('int32')
credits['납입회차'] = credits['납입회차'].astype('int32')
credits['연체기간'] = credits['연체기간'].astype('int32') # 구간칼럼은 모두 int32인데, 이것들은 int64임
credits['총납입금액'] = credits['총납입금액'].astype('int32')


# ! 구간회차 보정값 작성
    # 주민번호통합(다계좌) 월상환액이 있거나, 적용이율이 있으면 유예기간 상환액이 0이라도 유예기간을 스케줄에 반영해야하고
    # 그렇지 않은 경우는 유예기간을 스케줄에 반영하지 않아야함. 
    # 그래야 납입회차, 연체회차를 그대로 적용할 수 있으며, 스케줄에 반영된 경우에는 신복위에서도 0원으로 입금내역 보내줌
주민번호별첫상환액있음 = credits.groupby('신청인주민번호')['채무조정2월상환액1'].transform('max').gt(0)
이율있음 = credits['채무조정2적용이율1']>0
기존보정조건 = credits["유예기간"].gt(0) & credits["채무조정2월상환액1"].fillna(0).eq(0) & credits["채무조정2적용이율1"].fillna(0).eq(0)
credits["구간회차보정"] = credits["유예기간"].where(~주민번호별첫상환액있음 & 기존보정조건, 0).astype("int32")

스케줄작성대상 건수 : 942건
○ 신용회복키 없는 건 없음


### 전체 스케줄 작성

In [9]:
# 전체 스케줄 작성

구간스케줄칼럼 = []
for i in range(1,11) : 
    for v in ["시작회차","종료회차", "채무조정2월상환액"] :
        구간스케줄칼럼.append(f'{v}{i}')
스케줄생성필요칼럼 = ["신용회복키","상환개시일","유예기간","구간회차보정","납입회차","연체기간","총납입금액"] + 구간스케줄칼럼


스케줄칼럼 = ["신용회복키", "회차", "상환예정일", "상환액"]
전체스케줄 = pd.DataFrame(
    (행 for 당사스케줄 in credits[스케줄생성필요칼럼].apply(신용회복스케줄, axis=1) for 행 in 당사스케줄),
    columns=스케줄칼럼,
)

# 추가 작업
상태정보 = credits[["채무자키","계좌키", "신용회복키", "유예기간", "구간회차보정", "납입회차"]]
전체스케줄 = 전체스케줄.merge(상태정보, on="신용회복키", how="left", validate="many_to_one")
유예기간 = 전체스케줄["유예기간"] - 전체스케줄["구간회차보정"]

전체스케줄["상환액구분"] = np.select(
    [전체스케줄["회차"] < 0, 전체스케줄["회차"] <= 유예기간],
    ["미도래", "유예"],
    default="정상", # 정상 : 본회차(원금변제회차)
)
전체스케줄["상환여부"] = np.select(
    [
        전체스케줄["회차"] < 0,
        (전체스케줄["회차"] <= 전체스케줄["납입회차"]) & (전체스케줄["상환예정일"] <= 거래일),
        (전체스케줄["회차"] <= 전체스케줄["납입회차"]) & (전체스케줄["상환예정일"] > 거래일),
        전체스케줄["상환예정일"] <= 거래일,
    ],
    ["미도래", "납입", "선납입", "연체"],
    default="잔여",
)

전체스케줄 = 전체스케줄[["채무자키", "계좌키"] + 스케줄칼럼 + ["상환액구분", "상환여부"]]

# 시작회차가 n(>=2) 인 경우(누락포함 수정조정) 이전 무의미한 회차 삭제하기
삭제조건 = (전체스케줄.상환액구분=="정상") & 전체스케줄.상환액.eq(0)
전체스케줄 = 전체스케줄[~삭제조건]

### 건별 스케줄 조회

In [11]:
print('칼럼:',전체스케줄.columns)
print('상환액구분 종류:',전체스케줄.상환액구분.unique())
print('상환여부 종류:',전체스케줄.상환여부.unique())

# # 건별 조회
# 조회할열 = "신용회복키"
# 조회할값 = "21054833"
# pd.set_option("display.max_rows", 180)

# 전체스케줄[전체스케줄[조회할열] == 조회할값]

칼럼: Index(['채무자키', '계좌키', '신용회복키', '회차', '상환예정일', '상환액', '상환액구분', '상환여부'], dtype='object')
상환액구분 종류: ['정상' '유예' '미도래']
상환여부 종류: ['납입' '연체' '잔여' '미도래' '선납입']


In [41]:
# # 정상잔여분만 확인 (원금납입회차만)
# n = 9
# 정상잔여분 = 전체스케줄[(전체스케줄.상환액구분=="정상")&전체스케줄.상환여부.isin(["연체","잔여"])]


# # 순번별상환액 = 정상잔여분.sort_values(["신용회복키", "회차"]).copy() # 정렬되어 있으므로 불요
# df = (
#       정상잔여분.groupby("신용회복키").head(n)
#       .assign(순번=lambda x: x.groupby("신용회복키").cumcount() + 1)
#       .pivot(index="신용회복키", columns="순번", values="상환액")
#       .fillna(0) # 0은 상환완료 이후 회차
#       .assign(합계=lambda x: x.iloc[:, :n].sum(axis=1))
#       .reset_index()
#       )

# df.to_excel(join(r"C:\Users\DATA\Desktop",f'신용회복_정상회차분만_{n}회.xlsx'), index=False)

### 검증

In [112]:
####################
검증필요 = True
####################

if 검증필요 : 
    확정건 = credits[["채무자키","계좌키","신용회복키", "연체기간", "총납입금액"]].copy()
    키칼럼 = ["신용회복키"]

    스케줄연체회차 = (전체스케줄.loc[전체스케줄["상환여부"] == "연체"]
        .groupby(키칼럼).size().rename("스케줄연체회차").reset_index())
    연체회차검증 = 확정건[["채무자키","계좌키","신용회복키","연체기간"]].merge(스케줄연체회차, on=키칼럼, how="left")
    연체회차검증["스케줄연체회차"] = 연체회차검증["스케줄연체회차"].fillna(0).astype("int32")
    상환종기 = (전체스케줄.groupby(키칼럼)["상환예정일"].max().rename("상환종기").reset_index())
    연체회차검증 = 연체회차검증.merge(상환종기, on=키칼럼, how="left")

    def 상환종기이후회차수(상환종기):
        회차수 = 0
        while 상환종기 + pd.DateOffset(months=회차수 + 1) <= 거래일:
            회차수 += 1
        return 회차수

    연체회차검증["상환종기이후회차수"] = 연체회차검증["상환종기"].apply(상환종기이후회차수)
    연체회차검증["차이값"] = 연체회차검증["스케줄연체회차"] - 연체회차검증["연체기간"]
    상환종기차이일치 = (
        (연체회차검증["스케줄연체회차"] < 연체회차검증["연체기간"])
        & (연체회차검증["스케줄연체회차"] + 연체회차검증["상환종기이후회차수"] == 연체회차검증["연체기간"])
    )
    연체회차검증 = 연체회차검증.loc[(연체회차검증["차이값"] != 0) & ~상환종기차이일치].copy()

    스케줄납입금액 = (전체스케줄.loc[전체스케줄["상환여부"].str.contains('납입', na=False)]
        .groupby(키칼럼)["상환액"].sum().rename("스케줄납입금액").reset_index())
    납입금액검증 = 확정건[["채무자키",'계좌키',"신용회복키","총납입금액"]].merge(스케줄납입금액, on=키칼럼, how="left")
    납입금액검증["스케줄납입금액"] = 납입금액검증["스케줄납입금액"].fillna(0).astype("int64")
    납입금액검증["차이값"] = 납입금액검증["스케줄납입금액"] - 납입금액검증["총납입금액"]
    납입금액검증 = 납입금액검증.loc[납입금액검증["차이값"] != 0].copy()

    if 연체회차검증.empty & 납입금액검증.empty :
        print('○ 연체회차, 납입금액 모두 검증대상 없음')
    else : 
        path_검증결과 = join(r"C:\Users\DATA\Desktop", f"{cutoff}_신용회복_상환스케줄_검증결과.xlsx")
        with pd.ExcelWriter(path_검증결과, engine="openpyxl") as writer:
            출력대상 = {
                "연체회차검증": 연체회차검증,
                "납입금액검증": 납입금액검증,
            }
            for 시트명, 출력df in 출력대상.items():
                출력df.to_excel(writer, sheet_name=시트명, index=False)
                워크시트 = writer.sheets[시트명]
                for 열번호, 칼럼명 in enumerate(출력df.columns, start=1):
                    if pd.api.types.is_integer_dtype(출력df[칼럼명]):
                        for 행번호 in range(2, len(출력df) + 2):
                            워크시트.cell(행번호, 열번호).number_format = "#,##0"
                    elif pd.api.types.is_datetime64_any_dtype(출력df[칼럼명]):
                        for 행번호 in range(2, len(출력df) + 2):
                            워크시트.cell(행번호, 열번호).number_format = "yyyy-mm-dd"
        print(f'★ {path_검증결과}에 검증대상 출력')
        print(f'연체회차검증:{len(연체회차검증)}건, 납입금액검증:{len(납입금액검증)}건')
        
    # 납입/선납입 회차의 앞쪽 또는 그 사이에 다른 상태가 있는지 검증
    납입상태 = ["납입", "선납입"]
    납입구간종료회차 = (
        전체스케줄.loc[전체스케줄["상환여부"].isin(납입상태)]
        .groupby("신용회복키")["회차"].max()
        .rename("최종납입회차")
        .reset_index()
    )

    납입구간상태검증 = 전체스케줄.merge(납입구간종료회차, on="신용회복키", how="inner")
    납입구간상태검증 = 납입구간상태검증.loc[
        (납입구간상태검증["회차"] <= 납입구간상태검증["최종납입회차"])
        & ~납입구간상태검증["상환여부"].isin(납입상태)
    ].copy()

    if 납입구간상태검증.empty:
        print("○ 납입/선납입 구간의 상태 오류 없음")
    else:
        납입구간상태검증 = 납입구간상태검증.sort_values(["신용회복키", "회차"])
        path_납입구간상태검증 = join(r"C:\\Users\\DATA\\Desktop", f"{cutoff}_신용회복_납입구간상태검증.xlsx")
        납입구간상태검증.to_excel(path_납입구간상태검증, index=False)
        print(f"★ 납입/선납입 구간 상태 오류 {len(납입구간상태검증)}건: {path_납입구간상태검증}")
                                 
# 연체기간 차이값이 -인 경우 : 상환종기가 컷오프 이전이라 스케줄에 더이상 연체회차가 표시될 수 없는 경우 일것이다.
# 해당내용은 검증과정 반영완료

○ 연체회차, 납입금액 모두 검증대상 없음
○ 납입/선납입 구간의 상태 오류 없음


### 상환예정표 작성

In [117]:
# 월별 예정 스케줄 작성
# 출력유형이 "단순"이면 이 셀은 실행하지 않고, 상환예정표 작성 후 엑셀출력 셀로 이동한다.
##########################################
기준값 = "둘다"  # 현재회차, 납입회차, 둘다
조회할회차수 = 12
출력유형 = "단순"  # 상세: 입금·상태변경 포함 / 단순: 예정액만 출력 -> 바로 엑셀출력 셀로 
##########################################

# 전처리
if 기준값 not in ["현재회차", "납입회차", "둘다"]:
    raise ValueError('기준값은 현재회차, 납입회차, 둘다 중 하나여야 합니다.')
if 조회할회차수 < 1:
    raise ValueError('조회할회차수는 1 이상이어야 합니다.')
if 출력유형 not in ["상세", "단순"]:
    raise ValueError('출력유형은 상세 또는 단순 중 하나여야 합니다.')

대상키 = credits[["채무자키", "계좌키", "신용회복키"]].drop_duplicates("신용회복키").copy()

# 상환예정표 함수
def 상환예정표_작성(대상스케줄, 기준월기본값=None):
    """스케줄을 정렬·순번화한 뒤 pivot으로 가로형 예정표와 월별 기준정보를 만든다."""
    대상스케줄 = 대상스케줄.sort_values(["신용회복키", "회차"]).copy()
    대상스케줄["순번"] = 대상스케줄.groupby("신용회복키").cumcount().add(1)
    기간정보 = 대상스케줄.loc[대상스케줄["순번"] <= 조회할회차수].copy()
    기간정보["기간"] = 기간정보["상환예정일"].dt.to_period("M")

    예정표 = 대상키.copy()
    if 기준월기본값 is None:
        기준월 = (기간정보.drop_duplicates("신용회복키", keep="first")
                  .set_index("신용회복키")["기간"].astype(str))
        예정표["기준월"] = 예정표["신용회복키"].map(기준월).fillna("종료")
    else:
        예정표["기준월"] = 기준월기본값

    순번칼럼 = range(1, 조회할회차수 + 1)
    예정액 = (기간정보.pivot(index="신용회복키", columns="순번", values="상환액")
             .reindex(columns=순번칼럼, fill_value=0).fillna(0).astype("int64"))
    예정액.columns = [f"예정액{순번}" for 순번 in 순번칼럼]
    예정액구분 = (기간정보.pivot(index="신용회복키", columns="순번", values="상환액구분")
                 .reindex(columns=순번칼럼, fill_value="종료").fillna("종료"))
    예정액구분.columns = [f"예정액구분{순번}" for 순번 in 순번칼럼]

    예정표 = (예정표.merge(예정액, on="신용회복키", how="left")
                    .merge(예정액구분, on="신용회복키", how="left"))
    예정액칼럼 = 예정액.columns.tolist()
    예정액구분칼럼 = 예정액구분.columns.tolist()
    예정표[예정액칼럼] = 예정표[예정액칼럼].fillna(0).astype("int64")
    예정표[예정액구분칼럼] = 예정표[예정액구분칼럼].fillna("종료")

    return 예정표, 기간정보

# 상환예정표 작성
현재회차기준상환예정표 = pd.DataFrame()
납입회차기준상환예정표 = pd.DataFrame()
다음기준월 = 거래일.to_period("M") + 1

if 기준값 in ["현재회차", "둘다"]:
    현재회차대상 = 전체스케줄.loc[
        전체스케줄["상환예정일"].dt.to_period("M").between(다음기준월, 다음기준월 + 조회할회차수 - 1)
    ]
    현재회차기준상환예정표, 현재회차기간정보 = 상환예정표_작성(현재회차대상, str(다음기준월))

if 기준값 in ["납입회차", "둘다"]:
    납입상태 = ["납입", "선납입"]
    최종납입회차 = (전체스케줄.loc[전체스케줄["상환여부"].isin(납입상태)]
                  .groupby("신용회복키")["회차"].max().rename("최종납입회차"))
    납입회차대상 = 전체스케줄.merge(최종납입회차, on="신용회복키", how="left")
    납입회차대상 = 납입회차대상.loc[
        ~납입회차대상["상환여부"].isin(납입상태)
        & (납입회차대상["최종납입회차"].isna()
           | (납입회차대상["회차"] > 납입회차대상["최종납입회차"]))
    ].drop(columns="최종납입회차")
    납입회차기준상환예정표, 납입회차기간정보 = 상환예정표_작성(납입회차대상)

### 실제 입금내역, 신용회복 변동사항 첨부

In [115]:
##########################################
path_deposit = r"C:\\Users\\DATA\\Desktop"
fn_deposit = "입금조회새창_20261015_1108.xlsx"
유효입금자구분 = ["채무자"]
최근cutoff = "260930"
##########################################
path_lastest_credits = join(r"D:\3.자산\전산 dataset",company,최근cutoff,"parquet","[상태DC]신용회복_"+최근cutoff+".parquet")

# 파일 읽기 
    # 입금내역
deposit_cols = ["채무자키","계좌키","입금일","입금합계"]
deposit = pd.read_excel(join(path_deposit, fn_deposit), dtype=functions.deposit_dtype)
deposit = deposit[(deposit.입금구분=="신용회복")&(deposit.입금자구분.isin(유효입금자구분))][deposit_cols]
    # 최근 신용회복 전체데이터
lastest_credits_ori = pd.read_parquet(path_lastest_credits)
if path_credits_key != "" : 
    lastest_credits = lastest_credits_ori[lastest_credits_ori.신용회복키.isin(credits_key.신용회복키)].copy()
else : 
    lastest_credits = lastest_credits_ori.copy()

"""
처리내용
credits와 lastest_credits의 신용회복키를 비교하여 어느 한쪽에만 있는 경우 
채무자키,신용회복키,어느쪽에만 있는지 여부를 출력
"""

# 입금내역을 상환예정표에 붙이기
"""
처리내용
입금내역을 계좌키별, 월별로 합산하여 상환예정표에 merge하기 (칼럼명은 '2026-01'...)
"""

# 최근 신용회복데이터와 비교 후 실효,완제,재조정 내역 확인
"""
처리내용
필요칼럼 : 신용회복키, 진행구분, 실효/완제/합의서포기일자, 재조정접수일자, 조정후원금, 재조정이전납입금액
찰럼참고사항 : 진행구분에는 빈값은 없음. 일자는 빈값 있을 수 있으며 텍스트형식이라 타입변경 필요할 수 있음
credits와 lastest_credits를 신용회복키를 기준으로 비교하여
상환예정표에 '진행상태비고'(이하 비고),'진행상태발생일'(이하 발생일)열 작성
진행상태비고 (날짜가 없으면 빈값으로, 넘버는 조건의 우선순위임)
1. 
lastest_credits 진행구분이 실효 - 비고 : 실효, 발생일 : '실효/완제/합의서포기일자'
lastest_credits 진행구분이 완제 - 비고 : 완제, 발생일 : '실효/완제/합의서포기일자'
2. 
1에 해당하지 않는 것중
lastest_credits의 재조정접수일이 credits보다 나중인 경우 - 비고 : 재조정, 발생일 : '재조정접수일자'
* 재조정에 해당하는 경우 조정후원금 +조정후비용+ 재조정이전납입금액의 차액을 '재조정후총원금변동'열에 작성
"""

# 신용회복키 비교 ---------------------------------------------------------
상태변경필요칼럼 = [
    "신용회복키", "진행구분", "접수번호", "접수일", "실효/완제/합의서포기일자",
    "재조정접수일자", "조정후원금","조정후비용", "재조정이전납입금액",
]
누락칼럼 = set(상태변경필요칼럼) - set(lastest_credits.columns)
if 누락칼럼:
    raise KeyError(f"최근 신용회복 데이터에 없는 칼럼: {sorted(누락칼럼)}")

기준키 = credits[["채무자키", "신용회복키"]].drop_duplicates("신용회복키")
최종키 = lastest_credits[["채무자키", "신용회복키"]].drop_duplicates("신용회복키")
신용회복키비교 = 기준키.merge(최종키, on="신용회복키", how="outer", indicator=True, suffixes=("_최초", "_최종"))
신용회복키비교 = 신용회복키비교.loc[신용회복키비교["_merge"].ne("both")].copy()
신용회복키비교["존재구분"] = 신용회복키비교["_merge"].map({"left_only": "최초기준일만", "right_only": "최종기준일만"})
신용회복키비교 = 신용회복키비교[["채무자키_최초", "채무자키_최종", "신용회복키", "존재구분"]]
if 신용회복키비교.empty:
    print("○ 최초·최종 신용회복키가 모두 일치")
else:
    print(f"★ 한쪽에만 존재하는 신용회복키: {len(신용회복키비교)}건")
    display(신용회복키비교)

# 계좌키·월별 실제 입금액 ---------------------------------------------------
deposit["입금일"] = pd.to_datetime(deposit["입금일"], errors="coerce")
deposit["입금합계"] = pd.to_numeric(deposit["입금합계"].astype(str).str.replace(",", ""), errors="coerce").fillna(0)
기간별입금 = (deposit.dropna(subset=["입금일"]).assign(기간=lambda x: x["입금일"].dt.to_period("M"))
              .groupby(["계좌키", "기간"], as_index=False)["입금합계"].sum())

# 최초·최종 기준일 사이의 진행상태 변경 --------------------------------------
상태변경정보 = lastest_credits[상태변경필요칼럼].drop_duplicates("신용회복키").copy()
상태변경정보["진행구분"] = 상태변경정보["진행구분"].fillna("").astype(str).str.strip()
상태변경정보["실효/완제/합의서포기일자"] = pd.to_datetime(상태변경정보["실효/완제/합의서포기일자"], errors="coerce")
상태변경정보["접수일"] = pd.to_datetime(상태변경정보["접수일"], errors="coerce")
상태변경정보["재조정접수일자"] = pd.to_datetime(상태변경정보["재조정접수일자"], errors="coerce")

최초재조정일 = (credits[["신용회복키", "접수번호", "재조정접수일자", "조정후원금", "조정후비용", "재조정이전납입금액"]].drop_duplicates("신용회복키")
              .assign(재조정접수일자=lambda x: pd.to_datetime(x["재조정접수일자"], errors="coerce"))
              .rename(columns={"접수번호": "최초접수번호", "재조정접수일자": "최초재조정접수일자", "조정후원금": "최초조정후원금", "조정후비용": "최초조정후비용", "재조정이전납입금액": "최초재조정이전납입금액"}))
상태변경정보 = 상태변경정보.merge(최초재조정일, on="신용회복키", how="left", validate="one_to_one")
진행구분 = 상태변경정보["진행구분"]
재접수 = (상태변경정보["접수번호"].fillna("").astype(str).str.strip().ne("")) & (상태변경정보["최초접수번호"].fillna("").astype(str).str.strip().ne("")) & (상태변경정보["접수번호"].fillna("").astype(str).str.strip().ne(상태변경정보["최초접수번호"].fillna("").astype(str).str.strip()))
재조정 = 진행구분.eq("미확정") | 상태변경정보["재조정접수일자"].gt(상태변경정보["최초재조정접수일자"])
상태변경정보["진행상태비고"] = np.select(
    [재접수, 진행구분.eq("실효"), 진행구분.eq("완제"), 재조정],
    ["재접수", "실효", "완제", "재조정"],
    default="",
)
상태변경정보["진행상태발생일"] = 상태변경정보["실효/완제/합의서포기일자"]
상태변경정보.loc[상태변경정보["진행상태비고"].eq("재접수"), "진행상태발생일"] = 상태변경정보["접수일"]
상태변경정보.loc[상태변경정보["진행상태비고"].eq("재조정"), "진행상태발생일"] = 상태변경정보["재조정접수일자"]
재조정후총원금변동 = (
    pd.to_numeric(상태변경정보["조정후원금"], errors="coerce").fillna(0)
    + pd.to_numeric(상태변경정보["조정후비용"], errors="coerce").fillna(0)
    + pd.to_numeric(상태변경정보["재조정이전납입금액"], errors="coerce").fillna(0)
    - pd.to_numeric(상태변경정보["최초조정후원금"], errors="coerce").fillna(0)
    - pd.to_numeric(상태변경정보["최초조정후비용"], errors="coerce").fillna(0)
    - pd.to_numeric(상태변경정보["최초재조정이전납입금액"], errors="coerce").fillna(0)
).astype("int64")
상태변경정보["재조정후총원금변동"] = 재조정후총원금변동.where(상태변경정보["진행상태비고"].eq("재조정"))
상태변경정보 = 상태변경정보[["신용회복키", "진행상태비고", "진행상태발생일", "재조정후총원금변동"]]


계좌별변제액순서 = (기간별입금.sort_values(["계좌키", "기간"]).copy()
                  .assign(순번=lambda x: x.groupby("계좌키").cumcount().add(1))
                  .loc[lambda x: x["순번"] <= 조회할회차수])


def 변제액_및_상태_첨부(예정표):
    """계좌별 실제 입금월 순서와 신용회복키별 상태변경정보를 예정표에 붙인다."""
    순번칼럼 = range(1, 조회할회차수 + 1)
    변제액 = (계좌별변제액순서.pivot(index="계좌키", columns="순번", values="입금합계")
              .reindex(columns=순번칼럼, fill_value=0).fillna(0).astype("int64"))
    변제액.columns = [f"변제액{순번}" for 순번 in 순번칼럼]

    결과 = (예정표.merge(상태변경정보, on="신용회복키", how="left")
                  .merge(변제액, on="계좌키", how="left"))
    예정액칼럼 = [f"예정액{순번}" for 순번 in 순번칼럼]
    변제액칼럼 = 변제액.columns.tolist()
    결과[변제액칼럼] = 결과[변제액칼럼].fillna(0).astype("int64")
    결과[["진행상태비고"]] = 결과[["진행상태비고"]].fillna("")
    결과["재조정후총원금변동"] = 결과["재조정후총원금변동"].astype("object").where(결과["재조정후총원금변동"].notna(), "")
    결과["예정액합계(a)"] = 결과[예정액칼럼].sum(axis=1)
    결과["변제액합계(b)"] = 결과[변제액칼럼].sum(axis=1)
    결과["차액(b-a)"] = 결과["변제액합계(b)"] - 결과["예정액합계(a)"]

    고정칼럼 = [
        "채무자키", "계좌키", "신용회복키", "진행상태비고", "진행상태발생일",
        "재조정후총원금변동", "기준월", "예정액합계(a)", "변제액합계(b)", "차액(b-a)",
    ]
    예정액구분칼럼 = [f"예정액구분{순번}" for 순번 in 순번칼럼]
    return 결과[고정칼럼 + 예정액칼럼 + 변제액칼럼 + 예정액구분칼럼]


if not 현재회차기준상환예정표.empty:
    현재회차기준상환예정표 = 변제액_및_상태_첨부(현재회차기준상환예정표)
if not 납입회차기준상환예정표.empty:
    납입회차기준상환예정표 = 변제액_및_상태_첨부(납입회차기준상환예정표)


○ 최초·최종 신용회복키가 모두 일치


### 엑셀출력

In [ ]:
from copy import copy
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

##########################################
이중칼럼양식 = "둘다"  # a, b, 둘다  (비교 후 하나만 남기기)
틀고정셀 = "F3" if 출력유형 == "단순" else "G3"
##########################################

if 이중칼럼양식 not in ["a", "b", "둘다"]:
    raise ValueError('이중칼럼양식은 a, b, 둘다 중 하나여야 합니다.')

예정액칼럼 = [f"예정액{순번}" for 순번 in range(1, 조회할회차수 + 1)]
변제액칼럼 = [f"변제액{순번}" for 순번 in range(1, 조회할회차수 + 1)]
예정액구분칼럼 = [f"예정액구분{순번}" for 순번 in range(1, 조회할회차수 + 1)]


def 단순스케줄표_작성(예정표):
    """입금·상태변경 정보를 붙이지 않은 예정액 전용 출력표를 만든다."""
    결과 = 예정표.copy()
    결과["예정액합계(a)"] = 결과[예정액칼럼].sum(axis=1)
    return 결과[["채무자키", "계좌키", "신용회복키", "기준월", "예정액합계(a)"] + 예정액칼럼 + 예정액구분칼럼]


if 출력유형 == "단순":
    현재회차기준상환예정표 = 단순스케줄표_작성(현재회차기준상환예정표)
    납입회차기준상환예정표 = 단순스케줄표_작성(납입회차기준상환예정표)

출력대상 = {}
if not 현재회차기준상환예정표.empty:
    출력대상["현재회차기준"] = 현재회차기준상환예정표
if not 납입회차기준상환예정표.empty:
    출력대상["납입회차기준"] = 납입회차기준상환예정표
if not 출력대상:
    raise ValueError('출력할 상환예정표가 없습니다.')

고정칼럼 = (
    ["채무자키", "계좌키", "신용회복키", "기준월", "예정액합계(a)"]
    if 출력유형 == "단순"
    else [
        "채무자키", "계좌키", "신용회복키", "진행상태비고", "진행상태발생일",
        "재조정후총원금변동", "기준월", "예정액합계(a)", "변제액합계(b)", "차액(b-a)",
    ]
)


def 이중헤더_a_정보():
    """상위열: 예정액·변제액·예정액구분 / 하위열: 1~n."""
    그룹정보 = [("예정액", 예정액칼럼, "D9EAF7")]
    if 출력유형 == "상세":
        그룹정보.append(("변제액", 변제액칼럼, "D9EAD3"))
    그룹정보.append(("예정액구분", 예정액구분칼럼, "FFF2CC"))
    return 그룹정보


def 이중헤더_b_정보():
    """상위열: 기간1~n / 하위열: 예정액·변제액·예정액구분."""
    그룹정보 = []
    for 순번 in range(1, 조회할회차수 + 1):
        그룹정보.append((
            f"기간{순번}",
            [f"예정액{순번}"] + ([f"변제액{순번}"] if 출력유형 == "상세" else []) + [f"예정액구분{순번}"],
            "D9EAF7",
        ))
    return 그룹정보


def 테두리강조(워크시트, 시작열, 종료열):
    굵은선 = Side(style="medium", color="404040")
    for 행번호 in range(1, 워크시트.max_row + 1):
        시작셀 = 워크시트.cell(행번호, 시작열)
        종료셀 = 워크시트.cell(행번호, 종료열)
        시작셀.border = Border(left=굵은선, right=시작셀.border.right, top=시작셀.border.top, bottom=시작셀.border.bottom)
        종료셀.border = Border(left=종료셀.border.left, right=굵은선, top=종료셀.border.top, bottom=종료셀.border.bottom)


def 이중헤더_시트작성(워크시트, 원본df, 양식):
    그룹정보 = 이중헤더_a_정보() if 양식 == "a" else 이중헤더_b_정보()
    출력칼럼 = 고정칼럼 + [칼럼명 for _, 칼럼목록, _ in 그룹정보 for 칼럼명 in 칼럼목록]
    출력df = 원본df.reindex(columns=출력칼럼).copy()

    얇은선 = Side(style="thin", color="B7B7B7")
    기본채우기 = PatternFill("solid", fgColor="E7E6E6")
    상태채우기 = PatternFill("solid", fgColor="FCE4D6")
    합계채우기 = PatternFill("solid", fgColor="DDEBF7")
    가운데 = Alignment(horizontal="center", vertical="center", wrap_text=True)
    헤더글꼴 = Font(size=10, bold=True)
    본문글꼴 = Font(size=10)

    # 고정칼럼은 1행을 비워 두고 2행에 제목을 써서 필터가 항상 동작하게 한다.
    상태칼럼 = {"진행상태비고", "진행상태발생일", "재조정후총원금변동"}
    합계칼럼 = {"예정액합계(a)", "변제액합계(b)", "차액(b-a)"}
    for 열번호, 칼럼명 in enumerate(고정칼럼, start=1):
        채우기 = 상태채우기 if 칼럼명 in 상태칼럼 else (합계채우기 if 칼럼명 in 합계칼럼 else 기본채우기)
        상단값 = f"=SUBTOTAL(9,{get_column_letter(열번호)}3:{get_column_letter(열번호)}{len(출력df) + 2})" if 칼럼명 in {"재조정후총원금변동", "예정액합계(a)", "변제액합계(b)", "차액(b-a)"} else ""
        for 행번호, 값 in [(1, 상단값), (2, 칼럼명)]:
            셀 = 워크시트.cell(행번호, 열번호, 값)
            셀.fill, 셀.font, 셀.alignment = 채우기, 헤더글꼴, 가운데
            셀.border = Border(left=얇은선, right=얇은선, top=얇은선, bottom=얇은선)
            if 행번호 == 1 and 칼럼명 in {"재조정후총원금변동", "예정액합계(a)", "변제액합계(b)", "차액(b-a)"}:
                셀.number_format = "#,##0"

    현재열 = len(고정칼럼) + 1
    for 상위명, 칼럼목록, 색상 in 그룹정보:
        끝열 = 현재열 + len(칼럼목록) - 1
        워크시트.merge_cells(start_row=1, start_column=현재열, end_row=1, end_column=끝열)
        상위셀 = 워크시트.cell(1, 현재열, 상위명)
        상위셀.fill = PatternFill("solid", fgColor=색상)
        상위셀.font, 상위셀.alignment = 헤더글꼴, 가운데
        for 열번호, 칼럼명 in enumerate(칼럼목록, start=현재열):
            하위명 = re.search(r"\d+$", 칼럼명).group() if 양식 == "a" else 칼럼명.rstrip("1234567890")
            하위셀 = 워크시트.cell(2, 열번호, 하위명)
            하위셀.fill = PatternFill("solid", fgColor=색상)
            하위셀.font, 하위셀.alignment = 헤더글꼴, 가운데
            하위셀.border = Border(left=얇은선, right=얇은선, top=얇은선, bottom=얇은선)
        현재열 = 끝열 + 1

    if "진행상태발생일" in 출력df.columns:
        출력df["진행상태발생일"] = 출력df["진행상태발생일"].astype("object").where(출력df["진행상태발생일"].notna(), None)
    금액칼럼 = set(예정액칼럼 + 변제액칼럼 + ["예정액합계(a)", "변제액합계(b)", "차액(b-a)", "재조정후총원금변동"])
    날짜칼럼 = {"진행상태발생일"}
    for 행번호, 행값 in enumerate(dataframe_to_rows(출력df, index=False, header=False), start=3):
        for 열번호, 값 in enumerate(행값, start=1):
            칼럼명 = 출력칼럼[열번호 - 1]
            셀 = 워크시트.cell(행번호, 열번호, 값)
            셀.font = 본문글꼴
            셀.border = Border(left=얇은선, right=얇은선, top=얇은선, bottom=얇은선)
            if 칼럼명 in 금액칼럼:
                셀.number_format = "#,##0"
            elif 칼럼명 in 날짜칼럼:
                셀.number_format = "yyyy-mm-dd"

    # 상태변경 영역과 각 이중칼럼 상위열의 시작·끝을 굵은 직선으로 구분한다.
    if 출력유형 == "상세":
        테두리강조(워크시트, 고정칼럼.index("진행상태비고") + 1, 고정칼럼.index("재조정후총원금변동") + 1)
    현재열 = len(고정칼럼) + 1
    for _, 칼럼목록, _ in 그룹정보:
        테두리강조(워크시트, 현재열, 현재열 + len(칼럼목록) - 1)
        현재열 += len(칼럼목록)

    for 열번호, 칼럼명 in enumerate(출력칼럼, start=1):
        너비 = 14 if 칼럼명 in 금액칼럼 else 12
        if 칼럼명 in ["채무자키", "계좌키", "신용회복키"]:
            너비 = 15
        elif 칼럼명 in 상태칼럼:
            너비 = 16
        워크시트.column_dimensions[get_column_letter(열번호)].width = 너비
    워크시트.freeze_panes = 틀고정셀
    워크시트.auto_filter.ref = f"A2:{get_column_letter(len(출력칼럼))}{워크시트.max_row}"
    워크시트.row_dimensions[1].height = 22
    워크시트.row_dimensions[2].height = 22


출력양식목록 = ["a", "b"] if 이중칼럼양식 == "둘다" else [이중칼럼양식]
path_상환예정표 = join(r"C:\\Users\\DATA\\Desktop", f"{cutoff}_신용회복_상환예정표_{조회할회차수}회.xlsx")
with pd.ExcelWriter(path_상환예정표, engine="openpyxl") as writer:
    for 원본시트명, 출력df in 출력대상.items():
        for 양식 in 출력양식목록:
            시트명 = f"{원본시트명}_{양식.upper()}"
            워크시트 = writer.book.create_sheet(시트명)
            이중헤더_시트작성(워크시트, 출력df, 양식)
    if 출력유형 == "상세" and not 신용회복키비교.empty:
        신용회복키비교.to_excel(writer, sheet_name="신용회복키비교", index=False)

print(f"○ 상환예정표 출력: {path_상환예정표}")

○ 상환예정표 출력: C:\\Users\\DATA\\Desktop\251231_신용회복_상환예정표_12회.xlsx
